
### Race Results Analysis

In [0]:
 %run 
 "/databricks-course/Formula 1/includes/configurations.ipynb"

In [0]:
%run "/databricks-course/Formula 1/includes/common_functions"

In [0]:
drivers_df = spark.read.parquet(f"{processed_folder_path}/drivers_new") \
.withColumnRenamed("number", "driver_number") \
.withColumnRenamed("name", "driver_name") \
.withColumnRenamed("nationality", "driver_nationality") 

In [0]:
%fs
ls "dbfs:/mnt/formula1dlvb/processed/circuits/"


path,name,size,modificationTime
dbfs:/mnt/formula1dlvb/processed/circuits/_SUCCESS,_SUCCESS,0,1698241566000
dbfs:/mnt/formula1dlvb/processed/circuits/_committed_7042476529486792368,_committed_7042476529486792368,123,1698241565000
dbfs:/mnt/formula1dlvb/processed/circuits/_delta_log/,_delta_log/,0,1698241636000
dbfs:/mnt/formula1dlvb/processed/circuits/_started_7042476529486792368,_started_7042476529486792368,0,1698241561000
dbfs:/mnt/formula1dlvb/processed/circuits/part-00000-08ddd7f5-3333-4d5d-af6a-c8c374245491-c000.snappy.parquet,part-00000-08ddd7f5-3333-4d5d-af6a-c8c374245491-c000.snappy.parquet,209088,1698242928000
dbfs:/mnt/formula1dlvb/processed/circuits/part-00000-a256906b-7927-43eb-bc4d-5587a18d606a-c000.snappy.parquet,part-00000-a256906b-7927-43eb-bc4d-5587a18d606a-c000.snappy.parquet,7851,1698241638000
dbfs:/mnt/formula1dlvb/processed/circuits/part-00000-tid-7042476529486792368-ea599ccf-5bab-4459-9859-ebe9947a2fa3-12-1-c000.snappy.parquet,part-00000-tid-7042476529486792368-ea599ccf-5bab-4459-9859-ebe9947a2fa3-12-1-c000.snappy.parquet,7826,1698241564000
dbfs:/mnt/formula1dlvb/processed/circuits/part-00001-82354be8-798c-49f2-be63-8034ed5d6d13-c000.snappy.parquet,part-00001-82354be8-798c-49f2-be63-8034ed5d6d13-c000.snappy.parquet,208477,1698242928000


In [0]:
constructors_df = spark.read.parquet(f"{processed_folder_path}/constructors_new") \
.withColumnRenamed("name", "team") 

In [0]:
display(constructors_df)

constructorId,constructorRef,team,nationality
1,mclaren,McLaren,British
2,bmw_sauber,BMW Sauber,German
3,williams,Williams,British
4,renault,Renault,French
5,toro_rosso,Toro Rosso,Italian
6,ferrari,Ferrari,Italian
7,toyota,Toyota,Japanese
8,super_aguri,Super Aguri,Japanese
9,red_bull,Red Bull,Austrian
10,force_india,Force India,Indian


In [0]:
circuits_df = spark.read.parquet(f"{processed_folder_path}/circuits") \
.withColumnRenamed("location", "circuit_location") 

In [0]:
display(circuits_df)

result_id,race_id,driver_id,constructor_id,number,grid,position,position_text,position_order,points,laps,time,milliseconds,fastest_lap,rank,fastest_lap_time,fastest_lap_speed,ingestion_date
1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.3,2023-10-25T14:08:45.161+0000
2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,2023-10-25T14:08:45.161+0000
3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,2023-10-25T14:08:45.161+0000
4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,2023-10-25T14:08:45.161+0000
5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,2023-10-25T14:08:45.161+0000
6,18,6,3,8,13,6,6,6,3.0,57,\N,null,50,14,1:29.639,212.974,2023-10-25T14:08:45.161+0000
7,18,7,5,14,17,7,7,7,2.0,55,\N,null,22,12,1:29.534,213.224,2023-10-25T14:08:45.161+0000
8,18,8,6,1,15,8,8,8,1.0,53,\N,null,20,4,1:27.903,217.18,2023-10-25T14:08:45.161+0000
9,18,9,2,4,2,null,R,9,0.0,47,\N,null,15,9,1:28.753,215.1,2023-10-25T14:08:45.161+0000
10,18,10,7,12,18,null,R,10,0.0,43,\N,null,23,13,1:29.558,213.166,2023-10-25T14:08:45.161+0000


In [0]:
races_df = spark.read.parquet(f"{processed_folder_path}/races_new") \
.withColumnRenamed("name", "race_name") \
.withColumnRenamed("race_timestamp", "race_date") 

In [0]:
results_df = spark.read.parquet(f"{processed_folder_path}/results_new") \
.withColumnRenamed("time", "race_time") 

### Join Circuits to Results

In [0]:
race_circuits_df = races_df.join(circuits_df, races_df.circuit_id == circuits_df.race_id, "inner") \
.select(races_df.race_id, races_df.race_year, races_df.race_name, races_df.race_date)

In [0]:
race_results_df = results_df.join(race_circuits_df, results_df.race_id == race_circuits_df.race_id) \
                            .join(drivers_df, results_df.driver_id == drivers_df.driver_id) \
                            .join(constructors_df, results_df.constructor_id == constructors_df.constructorId)

In [0]:
from pyspark.sql.functions import current_timestamp


In [0]:
final_df = race_results_df.select("race_year", "race_name", "race_date", "driver_name", "driver_number", "driver_nationality",
                                 "team", "grid", "fastest_lap", "race_time", "points", "position") \
                          .withColumn("created_date", current_timestamp())

In [0]:
display(final_df.filter("race_year == 2020 and race_name == 'Abu Dhabi Grand Prix'").orderBy(final_df.points.desc()))

race_year,race_name,race_date,driver_name,driver_number,driver_nationality,team,grid,fastest_lap,race_time,points,position,created_date
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000
2020,Abu Dhabi Grand Prix,2020-12-13T13:10:00.000+0000,Max Verstappen,33,Dutch,Red Bull,1,14,1:36:28.645,25.0,1,2023-10-25T14:14:31.939+0000


In [0]:
final_df.write.mode("overwrite").saveAsTable("f1_presentation.race_results")